# 🧠 Hyperparameter Tuning with Keras Tuner


## 📋 Overview

I use Keras Tuner to search for good hyperparameter values instead of guessing them by hand. I define a model-building function where the number of Dense units and the learning rate are left as open hyperparameters, configure a `RandomSearch` tuner to explore that space, run the search, and then pull out the best configuration to build and evaluate a final model.

Coming from RF, this is the same move as sweeping a matching network or bias point across a range of values and measuring the result, rather than solving for the "correct" component value analytically — some settings just don't have a clean closed-form answer, so I search empirically and measure.

**What I cover:**
- 📥 Installing Keras Tuner and preparing MNIST
- 🏗️ Defining a model-building function with open hyperparameters
- 🔍 Configuring a `RandomSearch` tuner
- 🔄 Running the hyperparameter search
- 🎯 Retrieving the best hyperparameters and training a final model
- 🧪 Practice: revisiting the full pipeline


## 🧩 Theory

A **parameter** (weights, biases) is learned from data during training. A **hyperparameter** (number of units, learning rate, batch size) is set *before* training and controls how that learning happens — it isn't something gradient descent can solve for directly.

Hyperparameter search treats this as an optimization problem over a different space entirely:

$$
\lambda^* = \arg\max_{\lambda \in \Lambda} \ \mathbb{E}\big[\text{val\_accuracy}(f_\lambda)\big]
$$

where $\Lambda$ is the space of hyperparameter combinations (units × learning rate, in this notebook) and $f_\lambda$ is a model trained with that particular configuration. Since there's no gradient with respect to $\lambda$ the way there is for weights, I can't backprop my way to the optimum — I have to actually train multiple candidate models and compare them.

**Random search** samples configurations from $\Lambda$ at random rather than exhaustively trying every combination (a grid search) or using a smarter, feedback-driven strategy (Bayesian optimization):

$$
\lambda_i \sim \text{Uniform}(\Lambda), \quad i = 1, \ldots, N_{\text{trials}}, \qquad \lambda^* = \arg\max_i \ \text{val\_accuracy}(f_{\lambda_i})
$$

For the learning rate specifically, I sample **log-uniformly** rather than linearly:

$$
\eta \sim \text{LogUniform}(10^{-4}, 10^{-2}) \iff \log_{10}\eta \sim \text{Uniform}(-4, -2)
$$

This matters because learning rate effects scale multiplicatively, not additively — the jump from $10^{-4}$ to $10^{-3}$ matters as much as the jump from $10^{-3}$ to $10^{-2}$, so sampling uniformly on a linear scale would waste most trials clustered at the high end.

### 📡 Telecom analogy

| Keras Tuner concept | Signal processing / RF equivalent |
|---|---|
| Hyperparameter search space $\Lambda$ | The range of tunable component values in a matching network (capacitance, bias point) |
| `RandomSearch` | Randomly sampling settings across a sweep rather than testing every combination (grid search) |
| `objective='val_accuracy'` | The performance metric you're optimizing for (e.g. return loss, S11) |
| `executions_per_trial=2` | Repeating a measurement at the same setting to average out noise before comparing configurations |
| Log-uniform sampling for learning rate | Sweeping frequency logarithmically on a network analyzer rather than linearly, since effects span orders of magnitude |


## Part 1 — 📥 Setup: Installing Keras Tuner & Preparing MNIST

I install TensorFlow, Keras Tuner, and pin NumPy for compatibility, then load and normalize MNIST.

**A syntax quirk worth flagging:** `!pip install numpy<2.0.0` (no quotes) is technically broken shell syntax — the unquoted `<` is interpreted as input redirection by the shell, not a version constraint. The correct form would be `!pip install "numpy<2.0.0"`. I keep the line exactly as written since it's what the source material has, but it's worth knowing this particular line likely doesn't pin NumPy the way it looks like it should.


In [ ]:
!pip install tensorflow==2.16.2
!pip install keras-tuner==1.4.7
!pip install numpy<2.0.0


`sys.setrecursionlimit(100000)` raises Python's recursion ceiling — a defensive measure against recursion errors that can surface in complex model-building code or certain library internals.


In [ ]:
import sys

# Increase recursion limit to prevent potential issues
sys.setrecursionlimit(100000)


In [ ]:
# Step 2: Import necessary libraries
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.datasets import mnist
from tensorflow.keras.optimizers import Adam
import os
import warnings

# Suppress all Python warnings
warnings.filterwarnings('ignore')

# Set TensorFlow log level to suppress warnings and info messages
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # 0 = all logs, 1 = filter out INFO, 2 = filter out INFO and WARNING, 3 = ERROR only


In [ ]:
# Step 3: Load and preprocess the MNIST dataset
(x_train, y_train), (x_val, y_val) = mnist.load_data()
x_train, x_val = x_train / 255.0, x_val / 255.0

print(f'Training data shape: {x_train.shape}')
print(f'Validation data shape: {x_val.shape}')


## Part 2 — 🏗️ Defining a Model-Building Function with Hyperparameters

Rather than hardcoding the number of units and the learning rate, I express both as **open hyperparameters** using the `HyperParameters` object (`hp`). Keras Tuner calls `build_model(hp)` once per trial, each time substituting a different combination of values for `hp.Int('units', ...)` and `hp.Float('learning_rate', ...)`.


In [ ]:
# Define a model-building function

def build_model(hp):
    model = Sequential([
        Flatten(input_shape=(28, 28)),
        Dense(units=hp.Int('units', min_value=32, max_value=512, step=32), activation='relu'),
        Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG')),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


**Reading the search space:**

| Hyperparameter | Range | Sampling | Meaning |
|---|---|---|---|
| `units` | 32 to 512, step 32 | Linear | Width of the hidden Dense layer — 15 possible values |
| `learning_rate` | $10^{-4}$ to $10^{-2}$ | Log-uniform | Adam's step size — spans two orders of magnitude |


## Part 3 — 🔍 Configuring the RandomSearch Tuner

I hand `build_model` to `kt.RandomSearch`, tell it to optimize validation accuracy, cap the search at 10 trials, and run each trial twice (`executions_per_trial=2`) to average out training noise before comparing configurations.


In [ ]:
# Create a RandomSearch Tuner

tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=2,
    directory='my_dir',
    project_name='intro_to_kt'
)

# Display a summary of the search space
tuner.search_space_summary()


## Part 4 — 🔄 Running the Hyperparameter Search

`tuner.search(...)` actually runs the trials: for each of the 10 sampled hyperparameter combinations, it builds a model via `build_model(hp)`, trains it for 5 epochs on `(x_train, y_train)`, evaluates on `(x_val, y_val)`, and (since `executions_per_trial=2`) repeats that twice before recording the averaged result.


In [ ]:
# Run the hyperparameter search
tuner.search(x_train, y_train, epochs=5, validation_data=(x_val, y_val))

# Display a summary of the results
tuner.results_summary()


## Part 5 — 🎯 Retrieving and Using the Best Hyperparameters

I pull the top-ranked hyperparameter combination out of the completed search, print its values, then build a fresh model with exactly those settings and train it properly — on the full training set, for more epochs than any individual search trial used.


In [ ]:
# Step 1: Retrieve the best hyperparameters

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f""" 

The optimal number of units in the first dense layer is {best_hps.get('units')}. 

The optimal learning rate for the optimizer is {best_hps.get('learning_rate')}. 

""")

# Step 2: Build and Train the Model with Best Hyperparameters
model = tuner.hypermodel.build(best_hps)
model.fit(x_train, y_train, epochs=10, validation_split=0.2)

# Evaluate the model on the test set
test_loss, test_acc = model.evaluate(x_val, y_val)
print(f'Test accuracy: {test_acc}')


**Worth noting:** the search trials each trained for 5 epochs directly against `(x_val, y_val)` as validation data, but this final training run uses `validation_split=0.2` (carved out of `x_train`) instead, then evaluates against `x_val`/`y_val` at the end — a slightly different validation setup for the "real" training run than what the search itself used to rank configurations.


## 🧪 Practice: Revisiting the Pipeline

The source material repeats these same five steps as practice exercises with blank cells and hidden solutions. I fill in every one below. Most are identical to what I've already built; where a solution differs in a small but real way, I flag it rather than passing over it silently.

### Practice 1 — Setting up Keras Tuner

**Difference worth noting:** this solution's install line is just `!pip install keras-tuner` — no TensorFlow/NumPy version pins like Part 1 used, and no quoting issue since there's no version constraint here at all.


In [ ]:
!pip install keras-tuner

# Step 2: Import necessary libraries
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.datasets import mnist
from tensorflow.keras.optimizers import Adam

# Step 3: Load and preprocess the MNIST data set
(x_train, y_train), (x_val, y_val) = mnist.load_data()
x_train, x_val = x_train / 255.0, x_val / 255.0

# Print the shapes of the training and validation datasets
print(f'Training data shape: {x_train.shape}')
print(f'Validation data shape: {x_val.shape}')


### Practice 2 — Defining the model with hyperparameters

Identical pattern to Part 2.


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam
import keras_tuner as kt

# Step 1: Define a model-building function
def build_model(hp):
    model = Sequential([
        Flatten(input_shape=(28, 28)),
        Dense(units=hp.Int('units', min_value=32, max_value=512, step=32), activation='relu'),
        Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG')),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


### Practice 3 — Configuring the hyperparameter search

Identical pattern to Part 3.


In [ ]:
import keras_tuner as kt

# Step 1: Create a RandomSearch Tuner
tuner = kt.RandomSearch(
    build_model,  # Ensure 'build_model' function is defined from previous code
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=2,
    directory='my_dir',
    project_name='intro_to_kt'
)

# Display a summary of the search space
tuner.search_space_summary()


### Practice 4 — Running the hyperparameter search

Identical pattern to Part 4.


In [ ]:
# Step 1: Run the hyperparameter search

tuner.search(x_train, y_train, epochs=5, validation_data=(x_val, y_val))

# Display a summary of the results

tuner.results_summary()


### Practice 5 — Analyzing and using the best hyperparameters

**Difference worth noting:** this solution names its evaluation variables `val_loss`/`val_acc` and prints "Validation accuracy" instead of Part 5's `test_loss`/`test_acc`/"Test accuracy" — purely cosmetic (both evaluate against the same `(x_val, y_val)` split), but worth knowing the naming isn't consistent across the two versions of this exercise.


In [ ]:
# Step 1: Retrieve the best hyperparameters

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f""" 

The optimal number of units in the first dense layer is {best_hps.get('units')}. 

The optimal learning rate for the optimizer is {best_hps.get('learning_rate')}. 

""")

# Step 2: Build and train the model with best hyperparameters

model = tuner.hypermodel.build(best_hps)

model.fit(x_train, y_train, epochs=10, validation_split=0.2)

# Evaluate the model on the validation set

val_loss, val_acc = model.evaluate(x_val, y_val)

print(f'Validation accuracy: {val_acc}')


## 📊 Summary

| Concept | What I did | Why it matters |
|---|---|---|
| 📥 Setup | Installed Keras Tuner, loaded/normalized MNIST | Standard prep, plus one library specific to hyperparameter search |
| 🏗️ Model-building function | Expressed `units` and `learning_rate` as `hp.Int`/`hp.Float` | Turns fixed choices into a searchable space |
| 🔍 RandomSearch tuner | `objective='val_accuracy'`, `max_trials=10`, `executions_per_trial=2` | Randomly samples the search space instead of exhaustive grid search |
| 🔄 Running the search | `tuner.search(...)` trains + evaluates every sampled configuration | The actual empirical sweep across hyperparameter space |
| 🎯 Best hyperparameters | `get_best_hyperparameters` → `tuner.hypermodel.build` → full training run | Converts the winning search result into a properly trained final model |
| 🧪 Practice | Re-ran every step, filled in every blank | Surfaced small inconsistencies (variable naming, install line differences) between the two versions |

**Telecom throughline:** hyperparameter search is an empirical sweep, not an analytical solve — the same reason I'd sweep a matching network's component values or a receiver's bias point across a range and measure performance at each setting, rather than solving for the "correct" value directly.


## 🧪 Sandbox

Space to keep experimenting beyond the practice exercises:

- Swap `kt.RandomSearch` for `kt.BayesianOptimization` or `kt.Hyperband` and compare how many trials it takes to reach a similar best-validation-accuracy
- Add a third hyperparameter (e.g. dropout rate via `hp.Float('dropout', 0.0, 0.5, step=0.1)`) to the search space
- Increase `max_trials` and `executions_per_trial` and see how much the "best" configuration's ranking changes with more sampling
- Try a linear (non-log) sampling for `learning_rate` and compare how much of the search gets wasted at the high end of the range
- Fix the `!pip install numpy<2.0.0` quoting issue and confirm the version pin actually takes effect


In [ ]:
# 🧪 Sandbox — experiment here
